# C7-cnn-transfer — Practice p27 — Solution

**Type:** scenario analysis · **Difficulty:** advanced · **Concepts:** layer-freezing, requires-grad

*Reasoning is required. Coding is limited to committing three prediction tuples and running the marked verification experiment.*

A small convolution–BatchNorm–ReLU trunk starts in training mode, with trainable parameters and ordinary gradient tracking. Three teammates each change **one** control while keeping the other two at baseline:

1. `eval_case`: call `trunk.eval()`; parameters still have `requires_grad=True`, and gradient tracking remains enabled.
2. `frozen_case`: leave `trunk.train()` but set every parameter's `requires_grad=False`; ordinary gradient tracking remains enabled.
3. `inference_case`: leave `trunk.train()` and its parameters trainable, but run the forward pass inside `torch.inference_mode()`.

For each case, predict a tuple in this exact order:

```text
(output_has_gradient_graph, output_changes_from_baseline, running_buffers_change, output_is_inference_tensor)
```

Here “gradient graph” means `output.requires_grad`; “output changes” compares the case with the baseline forward from an identical fresh trunk; “running buffers” means BatchNorm's `running_mean` or `running_var` changes during the forward pass; and “inference tensor” means `torch.is_inference(output)`. Two of the three cases agree on the first three observables and are told apart only by the fourth, so decide it deliberately rather than by elimination. Commit exact `tuple[bool, bool, bool, bool]` identifiers `prediction_eval`, `prediction_frozen`, and `prediction_inference` **before** running the verifier. Then explain in 3–5 sentences why `.eval()`, parameter freezing, and `inference_mode()` are independent controls.

**Banned (zero points): calling `.backward()`, constructing an optimizer, changing the supplied trunk/input, running or reading the verifier before all predictions are committed, or assigning any observed/verdict value yourself.**


In [ ]:
prediction_eval: tuple[bool, bool, bool, bool] = (True, True, False, False)
prediction_frozen: tuple[bool, bool, bool, bool] = (False, False, True, False)
prediction_inference: tuple[bool, bool, bool, bool] = (False, False, True, True)

committed_predictions = [prediction_eval, prediction_frozen, prediction_inference]



Calling `.eval()` changes BatchNorm from current-batch statistics to its stored running statistics, so the output changes and the buffers stop updating, but parameter gradients remain enabled. Freezing parameters removes their route into an autograd graph; it does not switch the trunk out of training mode, so the numerical output matches the baseline and BatchNorm still updates its buffers. `inference_mode()` also removes the graph for this forward without changing module mode, which again leaves the training-mode output and buffer update intact. Thus module behavior, parameter trainability, and autograd recording are independent controls even though frozen and inference cases happen to produce the same three booleans here.


### MARKED VERIFICATION CELL

This cell refuses uncommitted or malformed tuples, builds every observation from fresh copies of the actual trunk, and reports agreement only.


In [ ]:
from copy import deepcopy

import torch
import torch.nn as nn

if len(committed_predictions) != 3:
    raise RuntimeError("commit exactly three prediction tuples")
if any(
    value is Ellipsis
    or not isinstance(value, tuple)
    or len(value) != 4
    or any(type(flag) is not bool for flag in value)
    for value in committed_predictions
):
    raise RuntimeError("each named prediction must be a committed tuple of four bools")

SEED = 20260804
torch.manual_seed(SEED)
base_trunk = nn.Sequential(
    nn.Conv2d(3, 5, kernel_size=3, padding=1, bias=False),
    nn.BatchNorm2d(5),
    nn.ReLU(),
)
x = torch.randn(4, 3, 7, 7)

def run_case(*, training, parameters_require_grad, inference):
    trunk = deepcopy(base_trunk)
    trunk.train(training)
    for parameter in trunk.parameters():
        parameter.requires_grad_(parameters_require_grad)
    bn = trunk[1]
    before_mean = bn.running_mean.clone()
    before_var = bn.running_var.clone()
    context = torch.inference_mode() if inference else torch.enable_grad()
    with context:
        output = trunk(x)
    buffers_change = not (
        torch.allclose(bn.running_mean, before_mean, atol=1e-7, rtol=0)
        and torch.allclose(bn.running_var, before_var, atol=1e-7, rtol=0)
    )
    is_inference = bool(torch.is_inference(output))
    return bool(output.requires_grad), output.detach().clone(), buffers_change, is_inference

baseline_graph, baseline_output, baseline_buffers, _ = run_case(
    training=True, parameters_require_grad=True, inference=False
)
case_settings = [
    dict(training=False, parameters_require_grad=True, inference=False),
    dict(training=True, parameters_require_grad=False, inference=False),
    dict(training=True, parameters_require_grad=True, inference=True),
]
observed = []
for settings in case_settings:
    graph, output, buffers, is_inference = run_case(**settings)
    output_changes = not torch.allclose(
        output, baseline_output, atol=1e-7, rtol=0
    )
    observed.append((graph, output_changes, buffers, is_inference))

agreement = [hand == seen for hand, seen in zip(committed_predictions, observed)]
# Agreement is reported as a single verdict: per-case feedback would let the three tuples be
# recovered by re-running rather than reasoned out (gate finding, plan 014).
print("all_agree:", all(agreement))



### Answer check


In [ ]:
assert committed_predictions == [
    (True, True, False, False),
    (False, False, True, False),
    (False, False, True, True),
]
assert baseline_graph is True and baseline_buffers is True
assert all(agreement)
